# DockBench — Inference trên **Kaggle**

1. **Add Data:** `duchvminh/results` (models + plots) — đã có sẵn  
2. **Add Data** hoặc **gdown:** demo 1–2 complex (`demo_inference/`, vài MB)  
3. `git clone` + `pip install`  
4. Score **1–2 complex**

**Settings:** GPU + **Internet**. Chạy **Run All** từ đầu.

## 1. Cấu hình

In [ ]:
from pathlib import Path

WORKDIR = Path("/kaggle/working")
REPO_URL = "https://github.com/nWoWolfpac/KhoaLuanTotNghiep.git"
BRANCH = "main"
CLONE_DIR = WORKDIR / "KhoaLuanTotNghiep"

MODEL_ID = "geoformerdock"
CHECKPOINT = None

# --- Results (weights + plots) — dataset Kaggle của bạn ---
KAGGLE_RESULTS_PATH = "/kaggle/input/datasets/duchvminh/results/results"
# Nếu None: tự tìm dưới /kaggle/input hoặc gdown Drive
DRIVE_RESULTS_FOLDER_ID = ""  # ví dụ: "1ItrUVa9PUFoiF_8khIOwb1OWHQURGrAp"
RESULTS_CACHE = WORKDIR / "results_from_drive"

# --- Demo 1–2 complex (.gninatypes) ---
# Add Data: datasets/duchvminh/demo-inference  HOẶC để None + điền DRIVE_DEMO_FOLDER_ID
KAGGLE_DEMO_DATA_ROOT = "/kaggle/input/datasets/duchvminh/demo-inference"
DRIVE_DEMO_FOLDER_ID = ""  # folder Drive chứa demo_inference/ (nếu chưa Add Data)
DEMO_CACHE = WORKDIR / "demo_from_drive"
DEMO_TYPES_FILE = "types/demo.types"
DEMO_COMPLEX_IDS = ["4kqp", "2ydt"]  # None = mọi dòng trong demo.types

## 2. Clone code + pip install

In [ ]:
import os
import sys
import json
import subprocess

def sh(cmd, check=True):
    print(">>>", cmd)
    return subprocess.run(cmd, shell=True, check=check)

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">>>", " ".join(cmd))
    subprocess.run(cmd, check=True)

clone = Path(CLONE_DIR)
if not (clone / ".git").is_dir():
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {clone}")
else:
    sh(f"cd {clone} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only", check=False)

if (clone / "code_docking" / "dockbench").is_dir():
    DOCK_ROOT = clone / "code_docking"
elif (clone / "dockbench").is_dir():
    DOCK_ROOT = clone
else:
    raise FileNotFoundError(f"Không thấy dockbench/ trong {clone}")

sys.path.insert(0, str(DOCK_ROOT))
os.chdir(DOCK_ROOT)
print("DOCK_ROOT =", DOCK_ROOT.resolve())

# Cài torch stack + molgrid; tránh upgrade numpy/pandas của Kaggle
pip_install("torch==2.2.2", "pytorch-ignite>=0.4.10", "PyYAML>=5.4", "tqdm>=4.62")
pip_install("molgrid")
pip_install("py3Dmol")
print("pip install xong (không cài lại pandas/numpy)")

## 3. Load **`results/`** (Kaggle Dataset hoặc gdown Drive)

Cần `models/.../best_model.pt` và `plots/.../compare_best_metrics.csv`.

In [ ]:
METRICS_CSV_REL = "log_dynamics/baseline_comparison/compare_best_metrics.csv"


def _discover_results_root() -> Path | None:
    """Tìm thư mục có models/*.pt dưới /kaggle/input."""
    inp = Path("/kaggle/input")
    if not inp.is_dir():
        return None
    best = None
    for models_dir in inp.rglob("models"):
        if not models_dir.is_dir():
            continue
        if list(models_dir.rglob("best_model.pt")):
            root = models_dir.parent
            if best is None or "duchvminh" in str(root):
                best = root
    return best


def _resolve_models_root(root: Path) -> Path:
    root = root.resolve()
    if (root / "models").is_dir() and list((root / "models").rglob("best_model.pt")):
        return root / "models"
    if list(root.rglob("best_model.pt")):
        return root
    raise FileNotFoundError(f"Không thấy best_model.pt dưới {root}")


def _resolve_plots_root(root: Path) -> Path:
    root = root.resolve()
    for base in [root, root / "plots"]:
        if (base / METRICS_CSV_REL).is_file():
            return base
    raise FileNotFoundError(f"Không thấy {METRICS_CSV_REL} dưới {root}")


def load_results_root() -> Path:
    if KAGGLE_RESULTS_PATH:
        p = Path(KAGGLE_RESULTS_PATH)
        if p.is_dir():
            return p.resolve()
        print(f"WARN: KAGGLE_RESULTS_PATH không tồn tại: {p}")

    found = _discover_results_root()
    if found is not None:
        print("Auto-discover results:", found)
        return found.resolve()

    if DRIVE_RESULTS_FOLDER_ID:
        pip_install("gdown")
        cache = Path(RESULTS_CACHE)
        marker = cache / ".ok"
        if not marker.is_file():
            if cache.exists():
                sh(f"rm -rf {cache}")
            cache.mkdir(parents=True, exist_ok=True)
            url = f"https://drive.google.com/drive/folders/{DRIVE_RESULTS_FOLDER_ID}"
            sh(f"gdown --folder {url} -O {cache} --remaining-ok", check=False)
            marker.write_text("ok")
        for sub in [cache, cache / "results"]:
            if sub.is_dir() and list(sub.rglob("best_model.pt")):
                return sub.resolve()
        return cache.resolve()

    raise FileNotFoundError(
        "Không thấy results. Add Data 'duchvminh/results' hoặc đặt KAGGLE_RESULTS_PATH."
    )


RESULTS_ROOT = load_results_root()
models_root = _resolve_models_root(RESULTS_ROOT)
plots_dir = _resolve_plots_root(RESULTS_ROOT)

print("RESULTS_ROOT =", RESULTS_ROOT)
print("models_root  =", models_root)
print("plots_dir    =", plots_dir)

pts = sorted(models_root.rglob("best_model.pt"))
print(f"\nbest_model.pt: {len(pts)}")
for p in pts[:12]:
    print(f"  {p.relative_to(models_root)}  ({p.stat().st_size / 1e6:.1f} MB)")
if len(pts) > 12:
    print(f"  ... +{len(pts) - 12} file")
if not pts:
    raise FileNotFoundError("Không có .pt trong models/")

## 4. Bảng benchmark (từ `plots/` đã tải ở mục 3)

Đọc CSV bằng `csv` (stdlib) — tránh lỗi pandas/numpy sau pip.

In [ ]:
import csv
import html
from IPython.display import HTML, display

csv_path = Path(plots_dir) / METRICS_CSV_REL
with csv_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
if not rows:
    raise ValueError(f"CSV rỗng: {csv_path}")

want_cols = [
    "display_model", "model",
    "best_c_index", "best_balanced_acc", "best_pearson_r", "best_pr_auc",
    "best_mae", "best_rmse",
]
cols = [c for c in want_cols if c in rows[0]]

def _as_float(val, default=float("-inf")):
    try:
        return float(val)
    except (TypeError, ValueError):
        return default

if "best_c_index" in cols:
    rows.sort(key=lambda r: _as_float(r.get("best_c_index")), reverse=True)

def _cell(val):
    return html.escape(str(val if val is not None else ""))

thead = "<tr>" + "".join(f"<th>{_cell(c)}</th>" for c in cols) + "</tr>"
tbody = "".join(
    "<tr>" + "".join(f"<td>{_cell(r.get(c, ''))}</td>" for c in cols) + "</tr>"
    for r in rows
)
display(HTML(
    "<table style='border-collapse:collapse'>"
    f"<thead>{thead}</thead><tbody>{tbody}</tbody></table>"
))
print(f"{len(rows)} dòng — {csv_path}")

## 5. Kiểm tra molgrid

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
try:
    import molgrid
    print("molgrid OK")
except ImportError as e:
    print("molgrid lỗi:", e, "— chạy ô 5b nếu cần")

In [ ]:
# 5b. Chỉ chạy nếu ô 5 báo lỗi import molgrid
# !conda install -y -c conda-forge molgrid

## 6. gninatyper (bỏ qua)

Demo dùng `.gninatypes` có sẵn — **không chạy ô này**.

In [ ]:
import shutil

def find_gninatyper():
    if GNINATYPER is not None and Path(GNINATYPER).is_file():
        return Path(GNINATYPER)
    for cand in [DOCK_ROOT / "tools" / "gninatyper", WORKDIR / "gninatyper"]:
        if cand.is_file():
            return cand
    w = shutil.which("gninatyper")
    return Path(w) if w else None

print("gninatyper:", find_gninatyper() or "⚠ cần tools/gninatyper")

## 7. Load checkpoint (từ `models/`)

In [ ]:
from typing import Optional, Tuple
from dockbench.models.registry import build_model, canonical_name
from dockbench.target_normalizer import TargetNormalizer


def _summary_near_ckpt(ckpt: Path) -> dict:
    for p in [ckpt.parent / "summary.json", ckpt.parent.parent / "summary.json"]:
        if p.is_file():
            return json.loads(p.read_text(encoding="utf-8"))
    return {}


def find_checkpoint(model_id: str) -> Path:
    if CHECKPOINT:
        p = Path(CHECKPOINT)
        if p.is_file():
            return p.resolve()
        raise FileNotFoundError(CHECKPOINT)
    hits = [p for p in models_root.rglob("best_model.pt") if model_id in p.as_posix()]
    if not hits:
        hits = list((models_root / model_id).rglob("best_model.pt"))
    if not hits:
        raise FileNotFoundError(f"Không tìm thấy best_model.pt cho {model_id}")
    return sorted(hits)[-1].resolve()


def load_model(ckpt_path: Path, model_name: str, input_dims: Tuple[int, int, int, int]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    name = canonical_name(model_name or payload.get("model", MODEL_ID))
    summary = _summary_near_ckpt(ckpt_path)
    gkw = None
    if name == "geoformerdock":
        gkw = {
            "max_pseudo_atoms": int(summary.get("max_pseudo_atoms", 12)),
            "num_transformer_layers": int(summary.get("num_transformer_layers", 2)),
            "uncertainty": bool(summary.get("geoformer_uncertainty", False)),
        }
    model = build_model(name, input_dims, affinity=True, flex=False, geoformer_kwargs=gkw)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device).eval()
    norm = None
    ns = payload.get("target_normalizer")
    if payload.get("normalize_targets") and ns:
        norm = TargetNormalizer()
        norm.mean, norm.std, norm.fitted = float(ns["mean"]), float(ns["std"]), True
    return model, device, norm, name


ckpt_path = find_checkpoint(MODEL_ID)
print("Checkpoint:", ckpt_path)

## 8. Tải / load demo data (1–2 complex)

Add Data `duchvminh/demo-inference` **hoặc** `DRIVE_DEMO_FOLDER_ID` (folder `demo_inference/` trên Drive).

In [ ]:
import shutil

def _discover_demo_root() -> Path | None:
    inp = Path("/kaggle/input")
    if not inp.is_dir():
        return None
    for name in ("demo-inference", "demo_inference", "demo"):
        for p in inp.rglob(name):
            if p.is_dir() and (p / "types" / "demo.types").is_file():
                return p
            if p.is_dir() and list(p.rglob("*.gninatypes")):
                return p
    return None


def load_demo_root() -> Path:
    if KAGGLE_DEMO_DATA_ROOT:
        p = Path(KAGGLE_DEMO_DATA_ROOT)
        if p.is_dir():
            return p.resolve()
        print(f"WARN: KAGGLE_DEMO_DATA_ROOT không tồn tại: {p}")

    found = _discover_demo_root()
    if found is not None:
        print("Auto-discover demo:", found)
        return found.resolve()

    if DRIVE_DEMO_FOLDER_ID:
        pip_install("gdown")
        cache = Path(DEMO_CACHE)
        marker = cache / ".ok"
        if not marker.is_file():
            if cache.exists():
                sh(f"rm -rf {cache}")
            cache.mkdir(parents=True, exist_ok=True)
            url = f"https://drive.google.com/drive/folders/{DRIVE_DEMO_FOLDER_ID}"
            sh(f"gdown --folder {url} -O {cache} --remaining-ok", check=False)
            marker.write_text("ok")
        for sub in [cache, cache / "demo_inference"]:
            if sub.is_dir() and list(sub.rglob("*.gninatypes")):
                return sub.resolve()
        return cache.resolve()

    raise FileNotFoundError(
        "Thiếu demo data. Add Data 'duchvminh/demo-inference' (pack: "
        "bash scripts/pack_demo_inference.sh 4kqp 2ydt) hoặc DRIVE_DEMO_FOLDER_ID."
    )


demo_root = load_demo_root()
types_path = demo_root / DEMO_TYPES_FILE
if not types_path.is_file():
    hits = list(demo_root.rglob("demo.types"))
    if not hits:
        raise FileNotFoundError(f"Không thấy demo.types dưới {demo_root}")
    types_path = hits[0]
    if types_path.parent.name == "types":
        demo_root = types_path.parent.parent

filter_ids = None
if DEMO_COMPLEX_IDS:
    filter_ids = {x.lower() for x in DEMO_COMPLEX_IDS}

DEMO_RUNS = []
for raw in types_path.read_text(encoding="utf-8", errors="replace").splitlines():
    line = raw.split("#", 1)[0].strip()
    if not line or line.startswith("#"):
        continue
    parts = line.split()
    if len(parts) < 4:
        continue
    rec_rel, lig_rel = Path(parts[-2]), Path(parts[-1])
    complex_id = rec_rel.parts[0].lower() if rec_rel.parts else ""
    if filter_ids and complex_id not in filter_ids:
        continue
    rec_src, lig_src = demo_root / rec_rel, demo_root / lig_rel
    if not rec_src.is_file() or not lig_src.is_file():
        raise FileNotFoundError(f"Thiếu file: {rec_src} hoặc {lig_src}")

    work = WORKDIR / "dockbench_work" / complex_id
    work.mkdir(parents=True, exist_ok=True)
    shutil.copy2(rec_src, work / "rec.gninatypes")
    shutil.copy2(lig_src, work / "lig.gninatypes")
    (work / "score.types").write_text(
        f"{parts[0]} {parts[1]} rec.gninatypes lig.gninatypes\n", encoding="utf-8"
    )
    DEMO_RUNS.append({
        "id": complex_id,
        "work": work,
        "rec_rel": str(rec_rel),
        "lig_rel": str(lig_rel),
        "label": parts[0],
        "affinity_label": parts[1],
    })

if not DEMO_RUNS:
    raise ValueError("Không có complex nào — kiểm tra DEMO_COMPLEX_IDS / demo.types")

print(f"demo_root = {demo_root}")
print(f"Chuẩn bị {len(DEMO_RUNS)} complex:")
for r in DEMO_RUNS:
    print(f"  {r['id']}: {r['rec_rel']} + {r['lig_rel']} -> {r['work']}")

In [ ]:
gmaker = molgrid.GridMaker(resolution=0.5, dimension=23.5)

# Dùng complex đầu để lấy INPUT_DIMS (cùng kiểu nguyên tử)
_probe = DEMO_RUNS[0]["work"]
_prov0 = molgrid.ExampleProvider(
    data_root=str(_probe), balanced=False, shuffle=False,
    default_batch_size=1, iteration_scheme=molgrid.IterationScheme.SmallEpoch,
    cache_structs=False,
)
_prov0.populate(str(_probe / "score.types"))
INPUT_DIMS = tuple(int(x) for x in gmaker.grid_dimensions(_prov0.num_types()))

model, device, normalizer, model_name = load_model(ckpt_path, MODEL_ID, INPUT_DIMS)
print(f"Model: {model_name} | INPUT_DIMS: {INPUT_DIMS}\n")

score_rows = []
for run in DEMO_RUNS:
    work = run["work"]
    prov = molgrid.ExampleProvider(
        data_root=str(work), balanced=False, shuffle=False,
        default_batch_size=1, iteration_scheme=molgrid.IterationScheme.SmallEpoch,
        cache_structs=False,
    )
    prov.populate(str(work / "score.types"))
    grid = torch.zeros((1,) + INPUT_DIMS, dtype=torch.float32, device=device)
    gmaker.forward(prov.next_batch(1), grid, random_translation=0.0, random_rotation=False)
    with torch.no_grad():
        pose_log, aff = model(grid)
    pose_prob = float(torch.exp(pose_log)[0, 1].item())
    aff_val = float(aff[0].item())
    if normalizer and normalizer.fitted:
        aff_val = float(normalizer.denormalize(aff)[0].item())
    score_rows.append({
        "complex": run["id"],
        "pose_label": run["label"],
        "affinity_label_types": run["affinity_label"],
        "pose_P_good": round(pose_prob, 4),
        "affinity_pK_pred": round(aff_val, 4),
    })
    print(
        f"[{run['id']}] Pose P(good)={pose_prob:.4f}  "
        f"Affinity pK(pred)={aff_val:.4f}  "
        f"(labels in .types: pose={run['label']}, aff={run['affinity_label']})"
    )

print("\n--- Tóm tắt ---")
for row in score_rows:
    print(row)

## 9. Xem 3D (tuỳ chọn)

Cần PDB từ RCSB — chỉ minh họa; pose score từ `.gninatypes` CrossDock.

In [ ]:
import urllib.request
import py3Dmol

# Chỉ vẽ complex đầu; đổi VIEW_COMPLEX hoặc lặp nếu cần
VIEW_COMPLEX = DEMO_RUNS[0]["id"].upper()  # PDB id RCSB thường viết hoa

work = next(r["work"] for r in DEMO_RUNS if r["id"] == VIEW_COMPLEX.lower())
pdb = work / f"{VIEW_COMPLEX}.pdb"
if not pdb.is_file():
    urllib.request.urlretrieve(
        f"http://files.rcsb.org/download/{VIEW_COMPLEX}.pdb", pdb
    )

rec, lig = work / "rec.pdb", work / "lig.pdb"
rec_lines, lig_lines = [], []
for line in pdb.read_text(errors="replace").splitlines(True):
    if line.startswith("ATOM"):
        rec_lines.append(line)
    elif line.startswith("HETATM"):
        lig_lines.append(line)
rec.write_text("".join(rec_lines))
lig.write_text("".join(lig_lines))

view = py3Dmol.view(width=700, height=450)
view.addModel(rec.read_text(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
if lig.exists() and lig.stat().st_size > 0:
    view.addModel(lig.read_text(), "pdb")
    view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo()
view.show()
print(f"3D view: {VIEW_COMPLEX} (ligand = HETATM trong PDB, có thể khác pose trong .gninatypes)")